In [0]:
dbutils.widgets.text("test_library_ref", "", "Test Library Ref")

test_library_ref = dbutils.widgets.get("test_library_ref")

if test_library_ref != "":
    # If the value looks like a version number (starts with digit), prepend package name
    if test_library_ref[0].isdigit():
        %pip install 'databricks-labs-dqx=={test_library_ref}'
    else:
        %pip install '{test_library_ref}'
else:
    %pip install databricks-labs-dqx

%restart_python

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.config import WorkspaceFileChecksStorageConfig, TableChecksStorageConfig
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.checks_serializer import ChecksNormalizer
from databricks.sdk import WorkspaceClient
import os
import yaml

In [0]:
dbutils.widgets.text("table", "", "Table Name")
dbutils.widgets.text("catalog", "lakehouse", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")

table = dbutils.widgets.get("table")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
rules_path = "/Workspace/Users/krzysiek@cegladanych.pl/databricks-free/lakehouse/src/etl/quality_rules/adddress.yml"

input_df = spark.sql(f"SELECT * FROM {catalog}.{schema}.{table}")

dq_engine = DQEngine(WorkspaceClient())

checks = dq_engine.load_checks(config=WorkspaceFileChecksStorageConfig(location=rules_path))

valid_df, quarantine_df = dq_engine.apply_checks_by_metadata_and_split(input_df, checks)

display(valid_df)

display(quarantine_df)

In [0]:
valid_and_quarantine_df = dq_engine.apply_checks_by_metadata(input_df, checks)

display(valid_and_quarantine_df)